# Étape 3 — Apprentissage Supervisé

Cas d'usage : **Prédiction du montant de la course (Régression)**

Variable cible : `total_amount`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

df = pd.read_parquet('../data/yellow_tripdata_features.parquet')
print(f'Shape : {df.shape}')

## 3.1 Préparation des features

In [ ]:
FEATURES = ['trip_distance', 'duree_course', 'passenger_count',
            'heure_journee', 'jour_semaine', 'est_weekend',
            'est_heure_pointe', 'est_trajet_aeroport',
            'heure_sin', 'heure_cos']
TARGET = 'total_amount'

df_model = df[FEATURES + [TARGET]].dropna()
X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f'Train : {X_train.shape} — Test : {X_test.shape}')

## 3.2 Modèle baseline — Régression Linéaire

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('=== Régression Linéaire ===')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.2f}')
print(f'MAE  : {mean_absolute_error(y_test, y_pred_lr):.2f}')
print(f'R²   : {r2_score(y_test, y_pred_lr):.4f}')

## 3.3 Modèle avancé — Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest ===')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.2f}')
print(f'MAE  : {mean_absolute_error(y_test, y_pred_rf):.2f}')
print(f'R²   : {r2_score(y_test, y_pred_rf):.4f}')

## 3.4 Comparaison et importance des features

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

feat_imp.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Importance des features (Random Forest)')

axes[1].scatter(y_test[:500], y_pred_rf[:500], alpha=0.3)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_xlabel('Valeurs réelles')
axes[1].set_ylabel('Prédictions')
axes[1].set_title('Réel vs Prédit')

plt.tight_layout()
plt.savefig('../figures/03_supervised.png', dpi=150)
plt.show()